# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a full walkthrough for loading and exploring the FAIR² dataset using the `mlcroissant` library. The steps include loading metadata and records, overviewing available data structures, extracting data, performing EDA, and visualizing key variables.

### Dataset Source
The dataset is described using a Croissant schema available at the following URL:


[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant pandas matplotlib

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title:", getattr(metadata, 'name', None))
print("Dataset Description:", getattr(metadata, 'description', None))


## 2. Data Overview
Review available record sets, their `@id`s, fields, and columns.

All entities are referenced by their `@id`, as recommended for Croissant datasets.

In [ ]:
# List all available record sets and their `@id`s
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}")

for idx, record_set in enumerate(record_sets):
    print(f"[{idx}] Record Set Name: {getattr(record_set, 'name', None)}")
    print(f"    Record Set @id: {getattr(record_set, '@id', None)}")
    # List fields in the record set
    fields = getattr(record_set, 'fields', [])
    print(f"    Fields ({len(fields)}):")
    for field in fields:
        print(f"        - Field name: {getattr(field, 'name', None)} | @id: {getattr(field, '@id', None)} | dataType: {getattr(field, 'data_type', None)}")
    # List columns and column ids if available
    columns = getattr(record_set, 'columns', [])
    if columns:
        print("    Columns:")
        for column in columns:
            print(f"        - Column: {getattr(column, 'name', None)} | @id: {getattr(column, '@id', None)}")
    print("")

## 3. Data Extraction
Load data from one or more record sets into Pandas DataFrames for analysis.

All loaded data uses entity `@id`s for selection.

In [ ]:
# Collect all record set @ids
record_set_ids = [getattr(rs, '@id', None) for rs in record_sets]
print(f"Record set @ids: {record_set_ids}")

dataframes = {}
for rs in record_sets:
    rs_id = getattr(rs, '@id', None)
    print(f"Loading records from record set: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records, columns: {df.columns.tolist()}")
        else:
            print("No records available for this record set.")
    except Exception as e:
        print(f"Error loading records for {rs_id}: {e}")

# Select the first record set with data for demonstration
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Selected main record set for analysis: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())
else:
    print("No data found in any record set.")

## 4. Exploratory Data Analysis (EDA)
Explore and process key fields—such as protein abundance, length, or mass—by filtering, normalization, or grouping. 

All references below use record set, field, or column `@id`s as available in the dataset.

In [ ]:
# We'll select example numeric and grouping fields by @id, 
# but you may want to adjust based on printed field/column names above.

df = dataframes[main_record_set_id]
# Example: try to select 'Abundance' or 'MW_kDa' if available, else pick first numeric
import numpy as np

# Choose a field for numeric analysis
numeric_candidates = ['Abundance', 'MW_kDa', 'TotalPeptides', 'Length', 'coverage_percent']
available_columns = df.columns.tolist()

# Find first available numeric column
numeric_field = None
for col in numeric_candidates:
    if col in available_columns:
        numeric_field = col
        break

if not numeric_field:
    # fallback: first column with numeric-looking content
    for col in available_columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

if numeric_field is None:
    print("No numeric field found for analysis.")
else:
    print(f"Numeric field selected: {numeric_field}")

    # Try to convert column to numeric, if not already
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    # Filter for values above a given threshold (e.g., >10)
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Pick a field for grouping if available.
    possible_group_fields = ['Gene', 'ProteinGroup', 'Sample', 'accession', 'description']
    group_field = None
    for col in possible_group_fields:
        if col in available_columns:
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field}, mean {numeric_field}:")
        print(grouped_df.head())
    else:
        print("No appropriate group field found.")

## 5. Visualization
Visualize the distribution of the numeric field and the effect of filtering and normalization. Adjust the field names if the data uses different columns.

In [ ]:
if numeric_field is not None and not filtered_df.empty:
    # Histogram before filtering
    plt.figure(figsize=(10,4))
    df[numeric_field].hist(bins=40)
    plt.title(f"Distribution of {numeric_field} (all data)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Histogram after filtering
    plt.figure(figsize=(10,4))
    filtered_df[numeric_field].hist(bins=20)
    plt.title(f"Distribution of {numeric_field} (> {threshold})")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Normalized field distribution
    plt.figure(figsize=(10,4))
    filtered_df[f"{numeric_field}_normalized"].hist(bins=20)
    plt.title(f"Normalized {numeric_field} (Filtered)")
    plt.xlabel(f"{numeric_field}_normalized")
    plt.ylabel("Count")
    plt.show()

    # Optional: barplot if grouped_df exists and group_field
    if 'grouped_df' in locals() and group_field:
        top_n = grouped_df.nlargest(10, numeric_field)
        plt.figure(figsize=(10,6))
        plt.bar(top_n[group_field], top_n[numeric_field])
        plt.title(f"Top {group_field} groups by mean {numeric_field}")
        plt.xticks(rotation=45, ha='right')
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² mass spectrometry dataset using the `mlcroissant` library, referencing all dataset entities by their `@id` as recommended for reproducibility.

- **Schema-driven workflow:** All access to record sets, fields, and columns was by their `@id`.
- **Initial exploration:** We listed available record sets and their fields, and inspected extracted DataFrame columns.
- **EDA & visualization:** We performed simple filtering, normalization, and grouping on selected fields, and visualized the results.

You may adapt and extend this notebook for deeper analysis by referencing any Croissant `@id` entities as shown above for rigorous, FAIR-compliant data science.